# Stage 3B.04 — full serial runRun this launch cell first with `SCENE="id"`. After 48/48 completes and the GPU is idle, change it to `SCENE="ood"` and rerun. Resume is safe.

In [ ]:
import os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; OUT=Path.home()/"stage3b"; N=Path.home()/"stage1-native"; SCENE="id"
PY=Path.home()/("venv-stage1-id/bin/python" if SCENE=="id" else "venv-stage1-ood/bin/python"); log=OUT/f"stage3b_{SCENE}.log"; pidfile=OUT/f"stage3b_{SCENE}.pid"
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":"1","MUJOCO_EGL_DEVICE_ID":"1","MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
if SCENE=="ood": env.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_stage3b","--config",str(R/"async_vla_benchmark/configs/stage3b.yaml"),"--manifest",str(OUT/"stage3b_object_layout_manifest.csv"),"--output-dir",str(OUT),"--scene",SCENE,"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched",SCENE,proc.pid,log)


In [ ]:
import csv,os
from pathlib import Path
OUT=Path.home()/"stage3b"; rows=list(csv.DictReader(open(OUT/"stage3b_episode_results.csv"))) if (OUT/"stage3b_episode_results.csv").exists() else []
print(f"new episodes {len({r['run_id'] for r in rows})}/144; remaining {144-len({r['run_id'] for r in rows})}")
for scene in ("id","ood"):
 p=OUT/f"stage3b_{scene}.pid"; pid=int(p.read_text()) if p.exists() else -1
 try: os.kill(pid,0); alive=True
 except OSError: alive=False
 print(scene,"alive=",alive)
 log=OUT/f"stage3b_{scene}.log"
 if log.exists(): print("\n".join(log.read_text(errors='replace').splitlines()[-8:]))